# AirQuery: reproduce the results
Run the cells in order. Everything is re-created from public data; nothing is read from anyone's Drive unless you turn it on.

In [ ]:
USE_DRIVE = False   # True keeps results across Colab sessions (needs your own Google Drive)
REPO = "https://github.com/tanujverkala/AirQuery.git"

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJ = "/content/drive/MyDrive/airquery"
    os.makedirs(PROJ, exist_ok=True)
    os.chdir(PROJ)
if not os.path.exists("run_pipeline.py"):
    !git clone {REPO} .
print("cwd:", os.getcwd())

In [ ]:
!pip install -q lightgbm

## 1. Data and baselines
Downloads the 7 station files and the weather forecasts, builds features, then the rolling-origin folds and baselines.

In [ ]:
!python run_pipeline.py
!python baselines.py

## 2. Model selection on validation folds only

In [ ]:
!python train_model.py --configs A,B
!python calibrate.py
!python calibrate_rolling.py
!python error_analysis.py

## 3. Final test (run once)

In [ ]:
!python train_model.py --final-test
!python calibrate_rolling.py --test
!python error_analysis.py --test

## 4. Website data

In [ ]:
!python prepare_app_data.py

## Optional: ablations shown on the Results page
These reproduce the extra rows of the Ablations table. Their numbers were copied by hand into `prepare_app_data.py`.

In [ ]:
!python train_model.py --configs B --drop none --target level --tag raw_
!python train_model.py --configs B --drop doy_sin,doy_cos,dow,wx_temperature_2m,wx_relative_humidity_2m,wx_wind_speed_10m,wx_precipitation,wx_wd_sin,wx_wd_cos --tag nowx_
!python train_model.py --configs B --target logratio --tag logr_